# 03 — DeBERTa-v3 fine-tuning + OOF probabilities

This notebook fine-tunes `microsoft/deberta-v3-base` for binary prompt-injection classification. It creates OOF probabilities on the development set and then trains one final model on all development examples.

**Run with Runtime → Change runtime type → GPU in Colab.**

**Outputs:**
- `artifacts/deberta_oof.csv`
- `artifacts/deberta_test.csv`
- `artifacts/deberta_model/`

The Hugging Face sequence-classification workflow uses `AutoModelForSequenceClassification`, `TrainingArguments`, and `Trainer`; the current documentation also supports `processing_class=tokenizer`.

> OOF training means this notebook fine-tunes one temporary model per fold plus one final model. With 3 folds that is 4 fine-tuning runs.

In [1]:
!pip -q install -U "transformers>=4.46,<5" "accelerate>=0.34,<2" "sentencepiece" "safetensors"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from pathlib import Path
import gc, inspect, os, shutil
import numpy as np
import pandas as pd
import torch
from google.colab import drive
drive.mount('/content/drive')

# Force mixed precision OFF at the accelerate/env level. accelerate can pick
# up a leftover "mixed_precision: fp16" config from Colab's environment even
# when TrainingArguments(fp16=False) is set, and DeBERTa-v3's disentangled
# attention is known to produce NaN logits under fp16 autocast on T4 GPUs.
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
os.environ["MIXED_PRECISION"] = "no"

from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

# ---- Paths ----
PROJECT_DIR = Path("/content/drive/MyDrive/softcom-prompt-injection")
PROC_DIR = PROJECT_DIR / "data" / "processed"
ART_DIR = PROJECT_DIR / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR = ART_DIR / "deberta_model"

# ---- Hyperparameters ----
MODEL_NAME = "microsoft/deberta-v3-base"
RANDOM_STATE = 42
N_FOLDS = 3
# 256 matches deberta-v3-base's position_buckets config and comfortably
# covers these prompts (well under 100 tokens); 384 was unnecessary here
# and sits on a known DeBERTa-v2/v3 relative-position edge case.
MAX_LENGTH = 256
NUM_EPOCHS = 2
# Lowered from 2e-5: deberta-v3-base's disentangled attention is more prone
# to NaN logits at 2e-5 with a small effective batch size (8) on T4 GPUs.
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
TRAIN_BATCH = 4
EVAL_BATCH = 8
GRAD_ACCUM_STEPS = 2
MAX_GRAD_NORM = 1.0

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. In Colab select Runtime > Change runtime type > GPU.")

# ---- Compatibility shim: detect which kwargs this transformers version supports ----
def _supports_kw(cls, kw):
    try:
        return kw in inspect.signature(cls.__init__).parameters
    except (ValueError, TypeError):
        return False

HAS_EVAL_STRATEGY       = _supports_kw(TrainingArguments, "eval_strategy")
HAS_EVALUATION_STRATEGY = _supports_kw(TrainingArguments, "evaluation_strategy")
HAS_PROCESSING_CLASS    = _supports_kw(Trainer, "processing_class")
HAS_OPTIM               = _supports_kw(TrainingArguments, "optim")

import transformers as _tf
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("transformers:", _tf.__version__)
print("supports eval_strategy      :", HAS_EVAL_STRATEGY)
print("supports evaluation_strategy:", HAS_EVALUATION_STRATEGY)
print("supports processing_class   :", HAS_PROCESSING_CLASS)

if not (HAS_EVAL_STRATEGY or HAS_EVALUATION_STRATEGY):
    raise RuntimeError("Unsupported transformers build: no eval strategy kwarg available.")

Mounted at /content/drive
GPU: Tesla T4
PyTorch: 2.11.0+cu128
transformers: 4.57.6
supports eval_strategy      : True
supports evaluation_strategy: False
supports processing_class   : True


In [3]:
dev = pd.read_csv(PROC_DIR / "dev.csv")
test = pd.read_csv(PROC_DIR / "test.csv")
y = dev["label"].to_numpy(dtype=int)

id2label = {0: "BENIGN", 1: "ATTACK"}
label2id = {"BENIGN": 0, "ATTACK": 1}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Tokenizer loaded:", MODEL_NAME)
print("dev:", dev.shape, " test:", test.shape)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Tokenizer loaded: microsoft/deberta-v3-base
dev: (824, 3)  test: (207, 3)


## PyTorch dataset wrapper

The tokenizer preserves the prompt text and truncates only at the model input limit. The lexical model and rule layer still receive the full prompt.

In [4]:
class PromptDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_length=384):
        self.encodings = tokenizer(
            list(map(str, texts)),
            truncation=True,
            max_length=max_length,
            padding=False,
        )
        self.labels = None if labels is None else np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: self.encodings[k][idx] for k in self.encodings}
        if self.labels is not None:
            item["labels"] = int(self.labels[idx])
        return item


def make_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )


def logits_to_probs(logits):
    """Softmax -> P(attack), with a safety net for NaN/Inf logits.

    DeBERTa-v3's disentangled attention can occasionally produce NaN/Inf
    logits during unstable training (see the LR/precision notes above).
    Rather than let a single bad batch crash the whole run, replace
    non-finite values with a neutral 0.5 and print a warning so the
    instability is visible instead of silent.
    """
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    if not np.isfinite(probs).all():
        n_bad = int((~np.isfinite(probs)).sum())
        print(
            f"[WARNING] {n_bad} non-finite probability value(s) detected — "
            "replacing with 0.5 so training/evaluation can continue. This "
            "means the model produced NaN/Inf logits (training instability); "
            "if this keeps happening, lower LEARNING_RATE further (e.g. 5e-6) "
            "or increase GRAD_ACCUM_STEPS for a larger effective batch."
        )
        probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0)
    return probs


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = logits[0] if isinstance(logits, tuple) else logits
    probs = logits_to_probs(logits)

    if len(np.unique(labels)) < 2:
        # can happen on a tiny/unlucky eval slice; avoid crashing on AUC
        return {"roc_auc": float("nan"), "pr_auc": float("nan"),
                "f1": float(f1_score(labels, probs >= 0.5, zero_division=0))}

    return {
        "roc_auc": float(roc_auc_score(labels, probs)),
        "pr_auc":  float(average_precision_score(labels, probs)),
        "f1":      float(f1_score(labels, probs >= 0.5, zero_division=0)),
    }


HAS_WARMUP_RATIO = _supports_kw(TrainingArguments, "warmup_ratio")

def training_args(output_dir, do_eval=True):
    """Version-agnostic TrainingArguments."""
    kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH,
        per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        save_strategy="no",
        logging_strategy="steps",
        logging_steps=50,
        report_to="none",
        fp16=False,
        bf16=False,
        dataloader_num_workers=2,
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
        remove_unused_columns=False,
    )
    if HAS_OPTIM:
        # adamw_torch is more numerically stable for DeBERTa's disentangled
        # attention than the fused/8-bit optimizers some transformers
        # versions default to.
        kwargs["optim"] = "adamw_torch"
    if HAS_WARMUP_RATIO:
        kwargs["warmup_ratio"] = 0.10
    else:
        # warmup_steps requires an integer step count, not a float ratio
        kwargs["warmup_steps"] = 10

    eval_value = "epoch" if do_eval else "no"
    if HAS_EVAL_STRATEGY:
        kwargs["eval_strategy"] = eval_value
    else:
        kwargs["evaluation_strategy"] = eval_value

    return TrainingArguments(**kwargs)

def make_trainer(model, args, train_ds, eval_ds=None, collator=None, metrics=None):
    """Version-agnostic Trainer constructor."""
    kwargs = dict(
        model=model,
        args=args,
        train_dataset=train_ds,
        data_collator=collator,
    )
    if eval_ds is not None:
        kwargs["eval_dataset"] = eval_ds
    if metrics is not None:
        kwargs["compute_metrics"] = metrics
    if HAS_PROCESSING_CLASS:
        kwargs["processing_class"] = tokenizer
    else:
        kwargs["tokenizer"] = tokenizer
    return Trainer(**kwargs)


def train_one_fold(train_texts, train_labels, valid_texts, valid_labels, fold_dir):
    set_seed(RANDOM_STATE)
    model = make_model()
    train_ds = PromptDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
    valid_ds = PromptDataset(valid_texts, valid_labels, tokenizer, MAX_LENGTH)
    collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

    trainer = make_trainer(
        model=model,
        args=training_args(fold_dir, do_eval=True),
        train_ds=train_ds,
        eval_ds=valid_ds,
        collator=collator,
        metrics=compute_metrics,
    )
    trainer.train()
    pred = trainer.predict(valid_ds)
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    probs = logits_to_probs(logits)
    return model, trainer, probs

## OOF fine-tuning

Each validation probability comes from a model that never trained on that validation example. These OOF probabilities become the inputs to the meta-classifier in notebook 04.

In [5]:
texts = dev["text"].astype(str).tolist()
oof = np.zeros(len(dev), dtype=np.float32)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, va_idx) in enumerate(skf.split(texts, y), start=1):
    print(f"\n===== DeBERTa Fold {fold}/{N_FOLDS} =====")
    fold_dir = ART_DIR / f"_deberta_fold_{fold}"
    if fold_dir.exists():
        shutil.rmtree(fold_dir)

    model, trainer, probs = train_one_fold(
        [texts[i] for i in tr_idx], y[tr_idx],
        [texts[i] for i in va_idx], y[va_idx],
        fold_dir,
    )
    oof[va_idx] = probs.astype(np.float32)
    print("Fold F1:", f1_score(y[va_idx], probs >= 0.5, zero_division=0))
    if len(np.unique(y[va_idx])) > 1:
        print("Fold ROC-AUC:", roc_auc_score(y[va_idx], probs))
    else:
        print("Fold ROC-AUC: n/a (single class in this fold's validation split)")

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    if fold_dir.exists():
        shutil.rmtree(fold_dir)

if not np.isfinite(oof).all():
    n_bad = int((~np.isfinite(oof)).sum())
    print(f"\n[WARNING] {n_bad} non-finite OOF value(s) remained — replacing with 0.5.")
    oof = np.nan_to_num(oof, nan=0.5, posinf=1.0, neginf=0.0)

print("\nOOF ROC-AUC:", roc_auc_score(y, oof))
print("OOF PR-AUC:", average_precision_score(y, oof))
pd.DataFrame({"row_id": dev["row_id"], "deberta_prob": oof}).to_csv(ART_DIR / "deberta_oof.csv", index=False)


===== DeBERTa Fold 1/3 =====


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Roc Auc,Pr Auc,F1
1,0.667700,0.447685,0.982133,0.985003,0.705882
2,0.443100,0.333934,0.993280,0.994562,0.957929


...

Epoch,Training Loss,Validation Loss,Roc Auc,Pr Auc,F1
1,0.673700,0.466654,0.992320,0.992817,0.705882
2,0.491100,0.323227,0.997920,0.998167,0.983498


Fold F1: 0.9834983498349835
Fold ROC-AUC: 0.99792

===== DeBERTa Fold 3/3 =====


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Roc Auc,Pr Auc,F1
1,0.669200,0.483173,0.983893,0.987405,0.704492
2,0.475100,0.324865,0.995329,0.997127,0.973684


Fold F1: 0.9736842105263158
Fold ROC-AUC: 0.9953288590604027

OOF ROC-AUC: 0.9953912397921306
OOF PR-AUC: 0.9963612372269867


## Final DeBERTa model

After OOF generation, train a fresh model on all development examples. No test labels are used here.

In [6]:
if FINAL_MODEL_DIR.exists():
    shutil.rmtree(FINAL_MODEL_DIR)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

set_seed(RANDOM_STATE)
final_model = make_model()
final_train_ds = PromptDataset(dev["text"], y, tokenizer, MAX_LENGTH)
final_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
final_trainer = make_trainer(
    model=final_model,
    args=training_args(ART_DIR / "_deberta_final_train", do_eval=False),
    train_ds=final_train_ds,
    collator=final_collator,
)
final_trainer.train()
final_trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

# Test inference through Trainer.predict
test_ds = PromptDataset(test["text"], labels=None, tokenizer=tokenizer, max_length=MAX_LENGTH)
pred = final_trainer.predict(test_ds)
logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
test_probs = logits_to_probs(logits).astype(np.float32)

pd.DataFrame({"row_id": test["row_id"], "deberta_prob": test_probs}).to_csv(ART_DIR / "deberta_test.csv", index=False)

print("Saved model:", FINAL_MODEL_DIR)
print("Saved OOF:", ART_DIR / "deberta_oof.csv")
print("Saved test predictions:", ART_DIR / "deberta_test.csv")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss
50,0.670600
100,0.457300
150,0.288900
200,0.217900


Saved model: /content/drive/MyDrive/softcom-prompt-injection/artifacts/deberta_model
Saved OOF: /content/drive/MyDrive/softcom-prompt-injection/artifacts/deberta_oof.csv
Saved test predictions: /content/drive/MyDrive/softcom-prompt-injection/artifacts/deberta_test.csv


In [7]:
assert pd.read_csv(ART_DIR / "deberta_oof.csv")["row_id"].tolist()  == dev["row_id"].tolist()
assert pd.read_csv(ART_DIR / "deberta_test.csv")["row_id"].tolist() == test["row_id"].tolist()
print("DeBERTa artifact alignment checks passed.")

DeBERTa artifact alignment checks passed.
